# 2.3 µm Iterative MFによる背景領域選択と人工プルーム二吸収帯評価

実メタンプルームを含むHISUIシーンに対して、次の順序で評価

1. **元シーンの2.3 µm帯にIterative Matched Filterを適用**し、実プルーム候補を背景統計から除外する
2. 2.3 µm帯のMF応答が背景中心に近く、空間的にも安定した場所を人工プルーム注入位置として自動選択する
3. 選んだ場所へ、MODTRAN絶対濃度LUTから作った相対放射輝度比を用いて人工プルームを注入する
4. 注入後のシーンに対して、1.6 µm帯と2.3 µm帯を**独立にIterative MF**で評価する
5. 2.3 µm帯の候補を1.6 µm帯が同一画素または近傍で支持するかを調べる
6. 既知の人工プルーム真値に対して、単一帯域と二吸収帯相互確認の性能を比較する

ここで「低い2.3 µm MF応答」は、最小の負値ではなく、**robust Z-scoreが0付近でメタンらしい正応答がない領域**と解釈。強い負の外れ値は、影・地表異常・校正誤差の可能性があるため注入場所には使わない。

主な評価対象は次の4つ。

- 1.6 µm帯MF単独
- 2.3 µm帯MF単独
- 二帯域の同一画素AND
- 2.3 µm候補を1.6 µm帯が近傍で支持する相互確認型MF

元シーンに実プルームがあるため、人工注入の評価は、元シーンで既に検出されていた候補を差し引いた**新規検出マスク**と、人工プルーム周辺の局所評価領域を用いて行う。

In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import binary_dilation, label, maximum_filter
from scipy.stats import rankdata
np.set_printoptions(precision=5, suppress=True)


## 1. 設定

In [ ]:
# 入力
ROI_CSV = r"E:\refit\all_roi_spectra.csv"
CH4_LUT_CSV = r"E:\refit\CH4b.csv"
OUTPUT_DIR = Path("./dual_window_background_injection_output")

# MODTRAN・HISUI
BACKGROUND_CH4_PPM = 1.8
FWHM_NM = 12.5
WINDOW_16 = (1580.0, 1750.0)
WINDOW_23 = (2100.0, 2450.0)
UAS_MAX_ENHANCEMENT_PPM = 0.5
UAS_STEP_PPM = 0.05

# Iterative MF 
MAX_ITERATIONS_BASELINE = 12
MAX_ITERATIONS_INJECTION = 10
EXCLUSION_Z_16 = 2.5
EXCLUSION_Z_23 = 2.5
EXCLUSION_DILATION_PIXELS = 2
MIN_BACKGROUND_FRACTION = 0.55
MIN_BACKGROUND_PIXELS = 300
CONVERGENCE_NEW_PIXEL_FRACTION = 2.5e-4
COVARIANCE_SHRINKAGE = 0.08
COVARIANCE_RIDGE_RELATIVE = 1e-8

# 人工プルーム注入場所の選択 
# Noneなら2.3 µm Iterative MFから自動選択。指定する場合はROI配列内の(row, col)
INJECTION_CENTER_YX = None
CENTER_SEARCH_STRIDE = 2
MAX_ABS_BASELINE_Z23 = 0.75
MAX_LOCAL_MAX_Z23 = 1.5
REAL_PLUME_EXCLUSION_BUFFER_PIXELS = 4

# 1.6 µm帯の既存異常が注入評価を汚さないための安全確認
# 主選択スコアは2.3 µm帯だけで作る
USE_16_AS_SAFETY_CHECK = False
MAX_ABS_BASELINE_Z16 = 1.5
MAX_LOCAL_ABS_Z16 = 2.0

# 人工プルーム形状 
PLUME_ANGLE_DEG = 0.0
PLUME_DECAY_PIX = 18.0
PLUME_CROSS_SIGMA_PIX = 4.0
PLUME_SOURCE_SIGMA_PIX = 2.0
PLUME_SUPPORT_FRACTION_FOR_LOCATION = 0.05

# LUT範囲外のピークは自動的に除外
INJECTION_PEAKS_PPM = [0.25, 0.5, 1.0, 1.5, 2.0, 3.0]
PRIMARY_PEAK_PPM = 2.0

# 最終候補抽出 
DETECTION_Z_16 = 2.0
DETECTION_Z_23 = 3.0
NEIGHBORHOOD_RADIUS = 1
MIN_REGION_PIXELS = 3
BASELINE_CANDIDATE_BUFFER_PIXELS = 2

# 人工真値・局所評価
TRUE_MASK_FRACTION_OF_PEAK = 0.10
TRUE_MASK_MIN_ENHANCEMENT_PPM = 0.05
EVALUATION_BUFFER_PIXELS = 12
TOP_LOCATION_CANDIDATES = 30

# 主ピークでしきい値探索
THRESHOLDS_16 = np.arange(0.5, 4.01, 0.5)
THRESHOLDS_23 = np.arange(1.5, 5.01, 0.5)

RANDOM_SEED = 42
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 2. HISUI ROIスペクトルCSVの読み込み

In [ ]:
def get_wave_columns(df):
    pattern = re.compile(r"^wave_([0-9.]+)nm$")
    pairs = []
    for col in df.columns:
        match = pattern.match(str(col))
        if match is not None:
            pairs.append((col, float(match.group(1))))
    if not pairs:
        raise ValueError("wave_***nm形式の列が見つかりません。")
    pairs.sort(key=lambda item: item[1])
    return [p[0] for p in pairs], np.array([p[1] for p in pairs], dtype=float)


def load_roi_spectra_csv(path):
    df = pd.read_csv(path)
    if "y" not in df.columns or "x" not in df.columns:
        raise ValueError("CSVには y と x 列が必要です。")
    wave_cols, wavelengths = get_wave_columns(df)
    spectra = df[wave_cols].to_numpy(dtype=float)
    return df, wavelengths, spectra


def spectra_to_cube(df, spectra, fill_value=np.nan):
    frame = df.reset_index(drop=True)
    ys = np.sort(frame["y"].unique())
    xs = np.sort(frame["x"].unique())
    y_to_i = {value: i for i, value in enumerate(ys)}
    x_to_i = {value: i for i, value in enumerate(xs)}

    cube = np.full((len(ys), len(xs), spectra.shape[1]), fill_value, dtype=float)
    for row_i, row in frame.iterrows():
        cube[y_to_i[row["y"]], x_to_i[row["x"]]] = spectra[row_i]
    return cube, ys, xs


def make_valid_pixel_mask(cube, nodata_values=(0.0, -9999.0), require_positive=True):
    valid = np.isfinite(cube)
    for value in nodata_values:
        valid &= cube != value
    if require_positive:
        valid &= cube > 0
    return np.all(valid, axis=2)


df, wavelengths, spectra = load_roi_spectra_csv(ROI_CSV)
cube_original, y_values, x_values = spectra_to_cube(df, spectra)
valid_mask = make_valid_pixel_mask(cube_original)

print("DataFrame shape:", df.shape)
print("Cube shape:", cube_original.shape)
print("Wavelength range:", wavelengths[0], "to", wavelengths[-1], "nm")
print("Valid pixels:", int(valid_mask.sum()), "/", valid_mask.size)


## 3. MODTRAN絶対濃度LUTの読み込み、HISUI応答への畳み込み、UAS作成


In [ ]:
def load_ch4_absolute_concentration_lut(path):
    df_lut = pd.read_csv(path)
    candidates = [
        col for col in df_lut.columns
        if str(col).strip().lower() in {"wavelength", "wave", "wavelength_nm", "waveln"}
    ]
    if not candidates:
        raise ValueError("LUTに波長列が見つかりません。")

    wave_col = candidates[0]
    mod_wave = df_lut[wave_col].to_numpy(dtype=float)

    pairs = []
    for col in df_lut.columns:
        if col == wave_col:
            continue
        try:
            pairs.append((col, float(str(col).strip())))
        except ValueError:
            pass
    if len(pairs) < 2:
        raise ValueError("LUTに絶対CH4濃度の数値列が2列以上必要です。")

    pairs.sort(key=lambda item: item[1])
    concentration_grid = np.array([p[1] for p in pairs], dtype=float)
    lut_spectra = df_lut[[p[0] for p in pairs]].to_numpy(dtype=float).T
    order = np.argsort(mod_wave)
    return mod_wave[order], concentration_grid, lut_spectra[:, order]


def gaussian_srf_resample(mod_wave, mod_spectra, sensor_wave, fwhm_nm):
    mod_wave = np.asarray(mod_wave, dtype=float)
    mod_spectra = np.asarray(mod_spectra, dtype=float)
    sensor_wave = np.asarray(sensor_wave, dtype=float)

    if np.isscalar(fwhm_nm):
        fwhm = np.full(sensor_wave.shape, float(fwhm_nm))
    else:
        fwhm = np.asarray(fwhm_nm, dtype=float)
    if fwhm.shape != sensor_wave.shape:
        raise ValueError("FWHM配列の長さがsensor_waveと一致しません。")

    output = np.full((mod_spectra.shape[0], sensor_wave.size), np.nan)
    for j, center in enumerate(sensor_wave):
        sigma = fwhm[j] / (2.0 * np.sqrt(2.0 * np.log(2.0)))
        use = np.abs(mod_wave - center) <= 4.0 * sigma
        if use.sum() < 2:
            output[:, j] = np.array([
                np.interp(center, mod_wave, spectrum)
                for spectrum in mod_spectra
            ])
        else:
            weights = np.exp(-0.5 * ((mod_wave[use] - center) / sigma) ** 2)
            weights /= weights.sum()
            output[:, j] = mod_spectra[:, use] @ weights
    return output


def interpolate_lut_spectrum(concentration, concentration_grid, sensor_lut):
    if not concentration_grid.min() <= concentration <= concentration_grid.max():
        raise ValueError(
            f"{concentration:.4f} ppmはLUT範囲外です。"
            f"LUT範囲: {concentration_grid.min():.4f}–{concentration_grid.max():.4f} ppm"
        )
    return np.array([
        np.interp(concentration, concentration_grid, sensor_lut[:, band_i])
        for band_i in range(sensor_lut.shape[1])
    ])


def compute_uas_from_absolute_lut(
    sensor_lut,
    concentration_grid,
    background_ppm,
    max_enhancement_ppm,
    step_ppm,
):
    max_allowed = concentration_grid.max() - background_ppm
    if max_enhancement_ppm > max_allowed:
        raise ValueError(
            f"UAS用増分がLUT範囲を超えます。使用可能最大値: {max_allowed:.4f} ppm"
        )

    enhancement_grid = np.arange(0.0, max_enhancement_ppm + 0.5 * step_ppm, step_ppm)
    enhancement_grid = np.unique(np.append(enhancement_grid, max_enhancement_ppm))

    background = interpolate_lut_spectrum(background_ppm, concentration_grid, sensor_lut)
    background = np.maximum(background, 1e-30)

    ratio_rows = []
    for enhancement in enhancement_grid:
        enhanced = interpolate_lut_spectrum(
            background_ppm + enhancement,
            concentration_grid,
            sensor_lut,
        )
        ratio_rows.append(enhanced / background)
    ratio_lut = np.asarray(ratio_rows)

    design = np.column_stack([np.ones_like(enhancement_grid), enhancement_grid])
    log_ratio = np.log(np.maximum(ratio_lut, 1e-30))
    coefficients, _, _, _ = np.linalg.lstsq(design, log_ratio, rcond=None)
    uas = -coefficients[1]
    return uas, enhancement_grid, ratio_lut


def build_enhancement_ratio_lut(
    sensor_lut,
    concentration_grid,
    background_ppm,
    maximum_enhancement_ppm,
    step_ppm=0.02,
):
    maximum_allowed = concentration_grid.max() - background_ppm
    if maximum_enhancement_ppm > maximum_allowed + 1e-12:
        raise ValueError(
            f"注入増分 {maximum_enhancement_ppm:.3f} ppm がLUT上限を超えます。"
            f"使用可能最大増分: {maximum_allowed:.3f} ppm"
        )

    grid = np.arange(0.0, maximum_enhancement_ppm + 0.5 * step_ppm, step_ppm)
    grid = np.unique(np.append(grid, maximum_enhancement_ppm))
    background = interpolate_lut_spectrum(background_ppm, concentration_grid, sensor_lut)
    background = np.maximum(background, 1e-30)

    rows = []
    for enhancement in grid:
        enhanced = interpolate_lut_spectrum(
            background_ppm + enhancement,
            concentration_grid,
            sensor_lut,
        )
        rows.append(enhanced / background)
    return grid, np.asarray(rows)


modtran_wavelengths, concentration_grid, modtran_spectra = (
    load_ch4_absolute_concentration_lut(CH4_LUT_CSV)
)
sensor_lut_absolute = gaussian_srf_resample(
    modtran_wavelengths,
    modtran_spectra,
    wavelengths,
    FWHM_NM,
)
uas_all, uas_enhancement_grid, uas_ratio_lut = compute_uas_from_absolute_lut(
    sensor_lut_absolute,
    concentration_grid,
    BACKGROUND_CH4_PPM,
    UAS_MAX_ENHANCEMENT_PPM,
    UAS_STEP_PPM,
)

mask_16 = (wavelengths >= WINDOW_16[0]) & (wavelengths <= WINDOW_16[1])
mask_23 = (wavelengths >= WINDOW_23[0]) & (wavelengths <= WINDOW_23[1])
if mask_16.sum() < 2 or mask_23.sum() < 2:
    raise ValueError(
        f"吸収窓内バンド数不足: 1.6 µm={mask_16.sum()}, 2.3 µm={mask_23.sum()}"
    )

maximum_allowed_enhancement = concentration_grid.max() - BACKGROUND_CH4_PPM
injection_peaks = np.array([
    value for value in INJECTION_PEAKS_PPM
    if 0 < value <= maximum_allowed_enhancement + 1e-12
], dtype=float)
if injection_peaks.size == 0:
    raise ValueError("LUT範囲内のINJECTION_PEAKS_PPMがありません。")
if PRIMARY_PEAK_PPM not in injection_peaks:
    PRIMARY_PEAK_PPM = float(injection_peaks[np.argmin(np.abs(injection_peaks - PRIMARY_PEAK_PPM))])
    warnings.warn(f"PRIMARY_PEAK_PPMをLUT範囲内の {PRIMARY_PEAK_PPM:.3f} ppmへ変更しました。")

enhancement_grid, enhancement_ratio_lut = build_enhancement_ratio_lut(
    sensor_lut_absolute,
    concentration_grid,
    BACKGROUND_CH4_PPM,
    float(injection_peaks.max()),
)

print("Absolute CH4 LUT grid [ppm]:", concentration_grid)
print("Injection peaks [ppm]:", injection_peaks)
print("Bands in 1.6 µm window:", int(mask_16.sum()))
print("Bands in 2.3 µm window:", int(mask_23.sum()))

plt.figure(figsize=(9, 4))
plt.plot(wavelengths[mask_16], uas_all[mask_16], label="1.6 µm UAS")
plt.plot(wavelengths[mask_23], uas_all[mask_23], label="2.3 µm UAS")
plt.xlabel("Wavelength [nm]")
plt.ylabel("UAS [1/ppm]")
plt.title("CH4 unit absorption spectrum after HISUI resampling")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


## 4. 単一帯域Iterative Matched Filter

各波長帯を独立に評価。現在の背景候補画素から平均と正則化共分散を求め、正の高応答画素を背景候補から単調に除外

$$
\alpha_i=
\frac{\boldsymbol{t}^{\mathsf T}\boldsymbol{\Sigma}^{-1}(\boldsymbol{x}_i-\boldsymbol{\mu})}
{\boldsymbol{t}^{\mathsf T}\boldsymbol{\Sigma}^{-1}\boldsymbol{t}},
\qquad
\boldsymbol{t}=-\boldsymbol{\mu}\odot\boldsymbol{s}
$$


In [ ]:
def robust_location_scale(values, floor=1e-12):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size < 2:
        raise ValueError("robust統計量の計算に必要な値が不足しています。")

    center = float(np.median(values))
    mad = float(np.median(np.abs(values - center)))
    scale = 1.4826 * mad
    if not np.isfinite(scale) or scale <= floor:
        scale = float(np.std(values, ddof=1))
    if not np.isfinite(scale) or scale <= floor:
        raise ValueError("MF背景分布の分散がほぼ0です。")
    return center, scale


def regularized_covariance(samples, shrinkage=0.08, ridge_relative=1e-8):
    samples = np.asarray(samples, dtype=float)
    if samples.ndim != 2 or samples.shape[0] < 2:
        raise ValueError("共分散推定用samplesの形状が不正です。")

    covariance = np.atleast_2d(np.cov(samples, rowvar=False, ddof=1))
    n_bands = covariance.shape[0]
    average_variance = float(np.trace(covariance) / max(n_bands, 1))
    if not np.isfinite(average_variance) or average_variance <= 0:
        diagonal = np.nanvar(samples, axis=0, ddof=1)
        good = diagonal[np.isfinite(diagonal) & (diagonal > 0)]
        average_variance = float(np.median(good)) if good.size else 1.0

    covariance = (
        (1.0 - shrinkage) * covariance
        + shrinkage * average_variance * np.eye(n_bands)
    )
    covariance += ridge_relative * max(average_variance, 1e-30) * np.eye(n_bands)
    return covariance


def fit_global_mf(
    cube,
    band_mask,
    uas,
    valid_mask,
    background_mask,
    covariance_shrinkage=0.08,
    covariance_ridge_relative=1e-8,
    minimum_background_pixels=300,
):
    band_cube = np.asarray(cube[:, :, band_mask], dtype=float)
    height, width, n_bands = band_cube.shape
    flat = band_cube.reshape(-1, n_bands)

    valid_flat = valid_mask.ravel()
    background_samples = flat[background_mask.ravel()]
    minimum_samples = max(minimum_background_pixels, 5 * n_bands)
    if background_samples.shape[0] < minimum_samples:
        raise ValueError(
            f"背景画素が不足しています: {background_samples.shape[0]} < {minimum_samples}"
        )

    background_mean = np.mean(background_samples, axis=0)
    covariance = regularized_covariance(
        background_samples,
        shrinkage=covariance_shrinkage,
        ridge_relative=covariance_ridge_relative,
    )
    target = -background_mean * np.asarray(uas[band_mask], dtype=float)

    try:
        inverse_cov_target = np.linalg.solve(covariance, target)
    except np.linalg.LinAlgError:
        warnings.warn("共分散行列のsolveに失敗したため疑似逆行列を使用します。")
        inverse_cov_target = np.linalg.pinv(covariance) @ target

    denominator = float(target @ inverse_cov_target)
    if not np.isfinite(denominator) or denominator <= 1e-30:
        raise ValueError("MF分母が不正です。UAS・波長窓・共分散を確認してください。")

    alpha_flat = np.full(flat.shape[0], np.nan, dtype=float)
    residual_valid = flat[valid_flat] - background_mean
    alpha_flat[valid_flat] = residual_valid @ inverse_cov_target / denominator
    alpha = alpha_flat.reshape(height, width)

    center, scale = robust_location_scale(alpha[background_mask])
    zscore = (alpha - center) / scale
    zscore[~valid_mask] = np.nan

    return {
        "alpha": alpha,
        "zscore": zscore,
        "background_mean": background_mean,
        "covariance": covariance,
        "target": target,
        "alpha_center": center,
        "alpha_scale": scale,
        "alpha_standard_error": 1.0 / np.sqrt(denominator),
        "denominator": denominator,
    }


def iterative_single_window_mf(
    cube,
    band_mask,
    uas,
    valid_mask,
    exclusion_z,
    max_iterations=12,
    dilation_pixels=2,
    minimum_background_fraction=0.55,
    convergence_new_pixel_fraction=2.5e-4,
    covariance_shrinkage=0.08,
    covariance_ridge_relative=1e-8,
    minimum_background_pixels=300,
    initial_excluded_mask=None,
    verbose=True,
):
    valid_count = int(valid_mask.sum())
    if initial_excluded_mask is None:
        excluded_mask = np.zeros_like(valid_mask, dtype=bool)
    else:
        excluded_mask = np.asarray(initial_excluded_mask, dtype=bool) & valid_mask
    background_mask = valid_mask & (~excluded_mask)
    history_rows = []

    for iteration in range(max_iterations):
        result = fit_global_mf(
            cube,
            band_mask,
            uas,
            valid_mask,
            background_mask,
            covariance_shrinkage,
            covariance_ridge_relative,
            minimum_background_pixels,
        )
        high_response_raw = valid_mask & (result["zscore"] >= exclusion_z)
        high_response = high_response_raw
        if dilation_pixels > 0:
            high_response = binary_dilation(
                high_response,
                iterations=dilation_pixels,
            ) & valid_mask

        proposed_excluded = excluded_mask | high_response
        proposed_background = valid_mask & (~proposed_excluded)
        minimum_background_count = max(
            minimum_background_pixels,
            int(np.ceil(minimum_background_fraction * valid_count)),
        )
        if int(proposed_background.sum()) < minimum_background_count:
            warnings.warn("背景画素が下限を下回るため、直前のマスクで停止します。")
            break

        newly_excluded = proposed_excluded & (~excluded_mask)
        new_fraction = float(newly_excluded.sum() / valid_count)
        history_rows.append({
            "iteration": iteration,
            "background_pixels_before_update": int(background_mask.sum()),
            "candidate_pixels_before_dilation": int(high_response_raw.sum()),
            "excluded_pixels_after_update": int(proposed_excluded.sum()),
            "newly_excluded_pixels": int(newly_excluded.sum()),
            "newly_excluded_fraction": new_fraction,
            "alpha_center": result["alpha_center"],
            "alpha_scale": result["alpha_scale"],
        })

        excluded_mask = proposed_excluded
        background_mask = proposed_background
        if verbose:
            print(
                f"iteration={iteration:02d}, background={int(background_mask.sum())}, "
                f"excluded={int(excluded_mask.sum())}, new={int(newly_excluded.sum())}"
            )
        if new_fraction <= convergence_new_pixel_fraction:
            if verbose:
                print("Converged.")
            break

    final_result = fit_global_mf(
        cube,
        band_mask,
        uas,
        valid_mask,
        background_mask,
        covariance_shrinkage,
        covariance_ridge_relative,
        minimum_background_pixels,
    )
    return {
        "result": final_result,
        "background_mask": background_mask,
        "excluded_mask": excluded_mask,
        "history": pd.DataFrame(history_rows),
    }


## 5. 元シーンのIterative MF

注入場所の選択は2.3 µm帯を主に使う。1.6 µm帯も元シーンで独立に実行し、人工注入前の既存応答を保存する。


In [ ]:
print("Baseline Iterative MF: 2.3 µm")
baseline_23 = iterative_single_window_mf(
    cube_original,
    mask_23,
    uas_all,
    valid_mask,
    exclusion_z=EXCLUSION_Z_23,
    max_iterations=MAX_ITERATIONS_BASELINE,
    dilation_pixels=EXCLUSION_DILATION_PIXELS,
    minimum_background_fraction=MIN_BACKGROUND_FRACTION,
    convergence_new_pixel_fraction=CONVERGENCE_NEW_PIXEL_FRACTION,
    covariance_shrinkage=COVARIANCE_SHRINKAGE,
    covariance_ridge_relative=COVARIANCE_RIDGE_RELATIVE,
    minimum_background_pixels=MIN_BACKGROUND_PIXELS,
)

print("\nBaseline Iterative MF: 1.6 µm")
baseline_16 = iterative_single_window_mf(
    cube_original,
    mask_16,
    uas_all,
    valid_mask,
    exclusion_z=EXCLUSION_Z_16,
    max_iterations=MAX_ITERATIONS_BASELINE,
    dilation_pixels=EXCLUSION_DILATION_PIXELS,
    minimum_background_fraction=MIN_BACKGROUND_FRACTION,
    convergence_new_pixel_fraction=CONVERGENCE_NEW_PIXEL_FRACTION,
    covariance_shrinkage=COVARIANCE_SHRINKAGE,
    covariance_ridge_relative=COVARIANCE_RIDGE_RELATIVE,
    minimum_background_pixels=MIN_BACKGROUND_PIXELS,
)

baseline_alpha_23 = baseline_23["result"]["alpha"]
baseline_z_23 = baseline_23["result"]["zscore"]
baseline_alpha_16 = baseline_16["result"]["alpha"]
baseline_z_16 = baseline_16["result"]["zscore"]

print("2.3 µm final background pixels:", int(baseline_23["background_mask"].sum()))
print("1.6 µm final background pixels:", int(baseline_16["background_mask"].sum()))


## 6. 2.3 µm帯から人工プルーム注入位置を選択

注入中心候補は、次を満たす場所から選ぶ。

- 2.3 µm Iterative MFの最終背景候補に含まれる
- 実プルーム候補とその周囲から離れている
- プルーム形状全体が有効画素内に収まる
- プルーム形状内の2.3 µm robust Z-scoreが0付近で、局所ばらつきも小さい
- 任意の安全確認として、1.6 µm帯に既存の強い異常がない

選択スコアは2.3 µm帯のみから計算する。


In [ ]:
def make_advected_enhancement_plume(
    shape,
    peak_enhancement_ppm,
    center_yx,
    angle_deg=0.0,
    decay_length=18.0,
    cross_sigma=4.0,
    source_sigma=2.0,
):
    height, width = shape
    center_y, center_x = center_yx
    yy, xx = np.meshgrid(np.arange(height), np.arange(width), indexing="ij")
    theta = np.deg2rad(angle_deg)
    dx = xx - center_x
    dy = yy - center_y
    along = dx * np.cos(theta) + dy * np.sin(theta)
    cross = -dx * np.sin(theta) + dy * np.cos(theta)
    downstream = np.maximum(along, 0.0)

    plume = (
        np.exp(-downstream / max(decay_length, 1e-6))
        * np.exp(-0.5 * (cross / max(cross_sigma, 1e-6)) ** 2)
    )
    plume *= along >= 0
    source = np.exp(-0.5 * (
        (dx / max(source_sigma, 1e-6)) ** 2
        + (dy / max(source_sigma, 1e-6)) ** 2
    ))
    plume = np.maximum(plume, source)
    if plume.max() > 0:
        plume /= plume.max()
    return peak_enhancement_ppm * plume


def plume_support_offsets(
    angle_deg,
    decay_length,
    cross_sigma,
    source_sigma,
    support_fraction=0.05,
):
    radius = int(np.ceil(max(4 * decay_length, 4 * cross_sigma, 4 * source_sigma)))
    size = 2 * radius + 1
    center = (radius, radius)
    template = make_advected_enhancement_plume(
        (size, size),
        1.0,
        center,
        angle_deg,
        decay_length,
        cross_sigma,
        source_sigma,
    )
    support = template >= support_fraction
    yy, xx = np.where(support)
    offsets = np.column_stack([yy - radius, xx - radius]).astype(int)
    weights = template[support]
    return offsets, weights, template


def choose_injection_location(
    alpha23,
    z23,
    z16,
    valid_mask,
    baseline_background_mask_23,
    baseline_excluded_mask_23,
    baseline_excluded_mask_16,
    offsets,
    stride=2,
    max_abs_center_z23=0.75,
    max_local_max_z23=1.5,
    exclusion_buffer_pixels=4,
    use_16_safety=True,
    max_abs_center_z16=1.5,
    max_local_abs_z16=2.0,
):
    forbidden = (~valid_mask) | binary_dilation(
        baseline_excluded_mask_23,
        iterations=exclusion_buffer_pixels,
    )
    if use_16_safety:
        forbidden |= binary_dilation(
            baseline_excluded_mask_16,
            iterations=exclusion_buffer_pixels,
        )
    height, width = valid_mask.shape
    rows = []
    score_map = np.full(valid_mask.shape, np.nan)

    for row in range(0, height, stride):
        for col in range(0, width, stride):
            if not baseline_background_mask_23[row, col]:
                continue
            if abs(z23[row, col]) > max_abs_center_z23:
                continue
            if use_16_safety and abs(z16[row, col]) > max_abs_center_z16:
                continue

            rr = row + offsets[:, 0]
            cc = col + offsets[:, 1]
            if rr.min() < 0 or cc.min() < 0 or rr.max() >= height or cc.max() >= width:
                continue
            if np.any(forbidden[rr, cc]):
                continue

            local_z23 = z23[rr, cc]
            local_z16 = z16[rr, cc]
            if not np.all(np.isfinite(local_z23)):
                continue
            if np.nanmax(local_z23) > max_local_max_z23:
                continue
            if use_16_safety:
                if not np.all(np.isfinite(local_z16)):
                    continue
                if np.nanmax(np.abs(local_z16)) > max_local_abs_z16:
                    continue

            local_median_abs_z23 = float(np.nanmedian(np.abs(local_z23)))
            local_std_z23 = float(np.nanstd(local_z23))
            # 主スコアは2.3 µmだけ。0付近かつ平坦な領域を優先。
            location_score = local_median_abs_z23 + 0.5 * local_std_z23
            score_map[row, col] = location_score

            rows.append({
                "row_index": int(row),
                "col_index": int(col),
                "center_z23": float(z23[row, col]),
                "center_alpha23": float(alpha23[row, col]),
                "center_z16": float(z16[row, col]),
                "local_median_abs_z16": float(np.nanmedian(np.abs(local_z16))),
                "local_max_abs_z16": float(np.nanmax(np.abs(local_z16))),
                "local_median_abs_z23": local_median_abs_z23,
                "local_std_z23": local_std_z23,
                "local_max_z23": float(np.nanmax(local_z23)),
                "location_score": location_score,
            })

    result = pd.DataFrame(rows)
    if result.empty:
        raise ValueError(
            "注入場所候補が見つかりません。MAX_ABS_BASELINE_Z23、"
            "MAX_LOCAL_MAX_Z23、PLUME_SUPPORT_FRACTION_FOR_LOCATION、"
            "REAL_PLUME_EXCLUSION_BUFFER_PIXELSを緩めてください。"
        )
    result = result.sort_values("location_score").reset_index(drop=True)
    return result, score_map


offsets, support_weights, plume_template = plume_support_offsets(
    PLUME_ANGLE_DEG,
    PLUME_DECAY_PIX,
    PLUME_CROSS_SIGMA_PIX,
    PLUME_SOURCE_SIGMA_PIX,
    PLUME_SUPPORT_FRACTION_FOR_LOCATION,
)

location_candidates_df, injection_location_score_map = choose_injection_location(
    baseline_alpha_23,
    baseline_z_23,
    baseline_z_16,
    valid_mask,
    baseline_23["background_mask"],
    baseline_23["excluded_mask"],
    baseline_16["excluded_mask"],
    offsets,
    stride=CENTER_SEARCH_STRIDE,
    max_abs_center_z23=MAX_ABS_BASELINE_Z23,
    max_local_max_z23=MAX_LOCAL_MAX_Z23,
    exclusion_buffer_pixels=REAL_PLUME_EXCLUSION_BUFFER_PIXELS,
    use_16_safety=USE_16_AS_SAFETY_CHECK,
    max_abs_center_z16=MAX_ABS_BASELINE_Z16,
    max_local_abs_z16=MAX_LOCAL_ABS_Z16,
)

if INJECTION_CENTER_YX is None:
    injection_center_yx = (
        int(location_candidates_df.iloc[0]["row_index"]),
        int(location_candidates_df.iloc[0]["col_index"]),
    )
else:
    injection_center_yx = tuple(map(int, INJECTION_CENTER_YX))
    manual_match = (
        (location_candidates_df["row_index"] == injection_center_yx[0])
        & (location_candidates_df["col_index"] == injection_center_yx[1])
    )
    if not manual_match.any():
        warnings.warn(
            "手動指定した注入中心は自動選択条件を満たしていません。"
            "実プルーム候補との重なり、有効画素、局所Z-scoreを確認してください。"
        )

center_row, center_col = injection_center_yx
print("Selected injection center [row, col]:", injection_center_yx)
print("Original coordinates [y, x]:", y_values[center_row], x_values[center_col])
print("Baseline Z23 at center:", baseline_z_23[center_row, center_col])
print("Baseline Z16 at center:", baseline_z_16[center_row, center_col])
display(location_candidates_df.head(TOP_LOCATION_CANDIDATES))

plt.figure(figsize=(6, 5))
plt.imshow(baseline_z_23)
plt.colorbar(label="Baseline robust Z: 2.3 µm")
plt.scatter(center_col, center_row, marker="x", s=100, label="Injection center")
plt.legend()
plt.title("Baseline 2.3 µm Iterative MF and selected injection center")
plt.xlabel("x index")
plt.ylabel("y index")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 5))
plt.imshow(injection_location_score_map)
plt.colorbar(label="Location score; lower is better")
plt.scatter(center_col, center_row, marker="x", s=100)
plt.title("Automatic injection-location suitability")
plt.xlabel("x index")
plt.ylabel("y index")
plt.tight_layout()
plt.show()


## 7. 人工メタンプルームの注入


In [ ]:
def interpolate_ratio_cube(enhancement_map, enhancement_grid, ratio_lut):
    flat = np.asarray(enhancement_map, dtype=float).reshape(-1)
    finite = np.isfinite(flat)
    if np.any(finite & ((flat < enhancement_grid.min()) | (flat > enhancement_grid.max()))):
        raise ValueError("濃度増分マップにLUT範囲外の値があります。")

    out = np.full((flat.size, ratio_lut.shape[1]), np.nan)
    for band_index in range(ratio_lut.shape[1]):
        out[finite, band_index] = np.interp(
            flat[finite],
            enhancement_grid,
            ratio_lut[:, band_index],
        )
    return out.reshape(enhancement_map.shape + (ratio_lut.shape[1],))


def inject_synthetic_methane_enhancement(
    cube_background,
    enhancement_map,
    enhancement_grid,
    ratio_lut,
    valid_mask,
):
    ratio_cube = interpolate_ratio_cube(enhancement_map, enhancement_grid, ratio_lut)
    injected = cube_background * ratio_cube
    injected[~valid_mask] = np.nan
    return injected, ratio_cube


def make_true_and_evaluation_masks(
    enhancement_map,
    valid_mask,
    peak_ppm,
    fraction_of_peak=0.10,
    minimum_enhancement_ppm=0.05,
    evaluation_buffer_pixels=12,
):
    threshold = max(minimum_enhancement_ppm, fraction_of_peak * peak_ppm)
    true_mask = valid_mask & np.isfinite(enhancement_map) & (enhancement_map >= threshold)
    evaluation_mask = binary_dilation(
        true_mask,
        iterations=evaluation_buffer_pixels,
    ) & valid_mask
    return true_mask, evaluation_mask, threshold


## 8. 検出マスクと評価指標

元シーンには実プルームがあるため、注入後の候補から元シーンで既に存在した候補とその周囲を差し引き、人工注入による**新規候補**を作る。評価は人工プルーム周辺の局所領域に限定。

閾値依存指標に加えて、1.6 µm帯が機能しているかを確認しやすいように、AUROC、平均適合率（AUPRC）、プルーム内外のZ-scoreコントラスト、ピーク位置誤差も出力する。


In [ ]:
def remove_small_components(binary_mask, minimum_pixels=1):
    component_labels, number_of_components = label(binary_mask)
    if number_of_components == 0:
        return np.zeros_like(binary_mask, dtype=bool)
    counts = np.bincount(component_labels.ravel())
    keep_ids = np.flatnonzero(counts >= minimum_pixels)
    keep_ids = keep_ids[keep_ids != 0]
    return np.isin(component_labels, keep_ids)


def neighborhood_supported_mask(primary_mask, support_score, support_threshold, radius=1):
    finite_support = np.where(np.isfinite(support_score), support_score, -np.inf)
    local_support_max = maximum_filter(
        finite_support,
        size=2 * radius + 1,
        mode="nearest",
    )
    return primary_mask & (local_support_max >= support_threshold), local_support_max


def binary_detection_metrics(true_mask, predicted_mask, evaluation_mask):
    truth = np.asarray(true_mask, dtype=bool) & evaluation_mask
    predicted = np.asarray(predicted_mask, dtype=bool) & evaluation_mask
    tp = int(np.sum(truth & predicted))
    fp = int(np.sum((~truth) & predicted & evaluation_mask))
    fn = int(np.sum(truth & (~predicted) & evaluation_mask))
    tn = int(np.sum((~truth) & (~predicted) & evaluation_mask))

    precision = tp / (tp + fp) if tp + fp else np.nan
    recall = tp / (tp + fn) if tp + fn else np.nan
    specificity = tn / (tn + fp) if tn + fp else np.nan
    fpr = fp / (fp + tn) if fp + tn else np.nan
    f1 = (
        2 * precision * recall / (precision + recall)
        if np.isfinite(precision) and np.isfinite(recall) and precision + recall > 0
        else np.nan
    )
    dice = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else np.nan
    iou = tp / (tp + fp + fn) if tp + fp + fn else np.nan
    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "false_positive_rate": fpr,
        "f1": f1,
        "dice": dice,
        "iou": iou,
    }


def rank_auc(y_true, score):
    y_true = np.asarray(y_true, dtype=bool)
    score = np.asarray(score, dtype=float)
    use = np.isfinite(score)
    y_true = y_true[use]
    score = score[use]
    n_pos = int(y_true.sum())
    n_neg = int((~y_true).sum())
    if n_pos == 0 or n_neg == 0:
        return np.nan
    ranks = rankdata(score, method="average")
    rank_sum_pos = float(ranks[y_true].sum())
    return (rank_sum_pos - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)


def average_precision_score_numpy(y_true, score):
    y_true = np.asarray(y_true, dtype=bool)
    score = np.asarray(score, dtype=float)
    use = np.isfinite(score)
    y_true = y_true[use]
    score = score[use]
    n_pos = int(y_true.sum())
    if n_pos == 0:
        return np.nan
    order = np.argsort(score)[::-1]
    sorted_true = y_true[order]
    cumulative_tp = np.cumsum(sorted_true)
    precision_at_k = cumulative_tp / np.arange(1, sorted_true.size + 1)
    return float(np.sum(precision_at_k[sorted_true]) / n_pos)


def continuous_score_metrics(true_mask, score, evaluation_mask):
    use = evaluation_mask & np.isfinite(score)
    truth = true_mask[use]
    values = score[use]
    return {
        "auroc": rank_auc(truth, values),
        "average_precision": average_precision_score_numpy(truth, values),
    }


def peak_localization_error(score, true_center_yx, evaluation_mask):
    masked = np.where(evaluation_mask & np.isfinite(score), score, -np.inf)
    if not np.any(np.isfinite(masked)) or np.nanmax(masked) == -np.inf:
        return np.nan, None
    peak_y, peak_x = np.unravel_index(np.argmax(masked), masked.shape)
    true_y, true_x = true_center_yx
    distance = float(np.hypot(peak_y - true_y, peak_x - true_x))
    return distance, (int(peak_y), int(peak_x))


def make_candidate_masks(z16, z23, valid_mask, threshold16, threshold23, radius, minimum_pixels):
    candidate_16 = remove_small_components(
        valid_mask & (z16 >= threshold16),
        minimum_pixels,
    )
    candidate_23 = remove_small_components(
        valid_mask & (z23 >= threshold23),
        minimum_pixels,
    )
    dual_exact = candidate_16 & candidate_23
    dual_nearby, local_max_z16 = neighborhood_supported_mask(
        candidate_23,
        z16,
        threshold16,
        radius,
    )
    return {
        "candidate_16": candidate_16,
        "candidate_23": candidate_23,
        "dual_exact": dual_exact,
        "dual_nearby": dual_nearby,
        "local_max_z16": local_max_z16,
    }


baseline_masks = make_candidate_masks(
    baseline_z_16,
    baseline_z_23,
    valid_mask,
    DETECTION_Z_16,
    DETECTION_Z_23,
    NEIGHBORHOOD_RADIUS,
    MIN_REGION_PIXELS,
)
baseline_buffer_16 = binary_dilation(
    baseline_masks["candidate_16"],
    iterations=BASELINE_CANDIDATE_BUFFER_PIXELS,
)
baseline_buffer_23 = binary_dilation(
    baseline_masks["candidate_23"],
    iterations=BASELINE_CANDIDATE_BUFFER_PIXELS,
)
baseline_buffer_dual = binary_dilation(
    baseline_masks["dual_nearby"],
    iterations=BASELINE_CANDIDATE_BUFFER_PIXELS,
)


## 9. 複数注入強度でのIterative MF評価

各ピーク増分について、元シーンへ人工プルームを加え、1.6 µm帯・2.3 µm帯を独立に再反復します。元シーンで除外された実プルーム候補は初期除外マスクとして引き継ぐ。

試行を軽くする場合は `INJECTION_PEAKS_PPM` を減らすか、`MAX_ITERATIONS_INJECTION` を小さくする。


In [ ]:
discrete_metric_rows = []
continuous_metric_rows = []
band_summary_rows = []
run_summary_rows = []
primary_result = None

for peak_ppm in injection_peaks:
    print("\n" + "=" * 72)
    print(f"Injected plume peak: {peak_ppm:.3f} ppm")

    enhancement_map = make_advected_enhancement_plume(
        cube_original.shape[:2],
        float(peak_ppm),
        injection_center_yx,
        PLUME_ANGLE_DEG,
        PLUME_DECAY_PIX,
        PLUME_CROSS_SIGMA_PIX,
        PLUME_SOURCE_SIGMA_PIX,
    )
    enhancement_map[~valid_mask] = np.nan
    cube_injected, ratio_cube = inject_synthetic_methane_enhancement(
        cube_original,
        enhancement_map,
        enhancement_grid,
        enhancement_ratio_lut,
        valid_mask,
    )
    true_mask, evaluation_mask, true_threshold = make_true_and_evaluation_masks(
        enhancement_map,
        valid_mask,
        float(peak_ppm),
        TRUE_MASK_FRACTION_OF_PEAK,
        TRUE_MASK_MIN_ENHANCEMENT_PPM,
        EVALUATION_BUFFER_PIXELS,
    )

    result_16 = iterative_single_window_mf(
        cube_injected,
        mask_16,
        uas_all,
        valid_mask,
        exclusion_z=EXCLUSION_Z_16,
        max_iterations=MAX_ITERATIONS_INJECTION,
        dilation_pixels=EXCLUSION_DILATION_PIXELS,
        minimum_background_fraction=MIN_BACKGROUND_FRACTION,
        convergence_new_pixel_fraction=CONVERGENCE_NEW_PIXEL_FRACTION,
        covariance_shrinkage=COVARIANCE_SHRINKAGE,
        covariance_ridge_relative=COVARIANCE_RIDGE_RELATIVE,
        minimum_background_pixels=MIN_BACKGROUND_PIXELS,
        initial_excluded_mask=baseline_16["excluded_mask"],
        verbose=False,
    )
    result_23 = iterative_single_window_mf(
        cube_injected,
        mask_23,
        uas_all,
        valid_mask,
        exclusion_z=EXCLUSION_Z_23,
        max_iterations=MAX_ITERATIONS_INJECTION,
        dilation_pixels=EXCLUSION_DILATION_PIXELS,
        minimum_background_fraction=MIN_BACKGROUND_FRACTION,
        convergence_new_pixel_fraction=CONVERGENCE_NEW_PIXEL_FRACTION,
        covariance_shrinkage=COVARIANCE_SHRINKAGE,
        covariance_ridge_relative=COVARIANCE_RIDGE_RELATIVE,
        minimum_background_pixels=MIN_BACKGROUND_PIXELS,
        initial_excluded_mask=baseline_23["excluded_mask"],
        verbose=False,
    )

    alpha16 = result_16["result"]["alpha"]
    alpha23 = result_23["result"]["alpha"]
    z16 = result_16["result"]["zscore"]
    z23 = result_23["result"]["zscore"]
    raw_masks = make_candidate_masks(
        z16,
        z23,
        valid_mask,
        DETECTION_Z_16,
        DETECTION_Z_23,
        NEIGHBORHOOD_RADIUS,
        MIN_REGION_PIXELS,
    )

    # 元シーンで既に検出されていた候補を差し引く。
    new16 = remove_small_components(
        raw_masks["candidate_16"] & (~baseline_buffer_16),
        MIN_REGION_PIXELS,
    )
    new23 = remove_small_components(
        raw_masks["candidate_23"] & (~baseline_buffer_23),
        MIN_REGION_PIXELS,
    )
    new_exact = new16 & new23
    support_z16 = np.where(~baseline_buffer_16, z16, -np.inf)
    new_nearby, local_max_new_z16 = neighborhood_supported_mask(
        new23,
        support_z16,
        DETECTION_Z_16,
        NEIGHBORHOOD_RADIUS,
    )
    new_nearby &= ~baseline_buffer_dual

    score_maps = {
        "1.6 µm Z": z16,
        "2.3 µm Z": z23,
        "Dual minimum Z": np.minimum(z16, z23),
        "Dual geometric Z": np.sqrt(
            np.clip(z16, 0.0, None) * np.clip(z23, 0.0, None)
        ),
    }
    method_masks = {
        "1.6 µm only": new16,
        "2.3 µm only": new23,
        "Dual exact": new_exact,
        "Dual nearby": new_nearby,
    }

    for method, predicted in method_masks.items():
        discrete_metric_rows.append({
            "peak_enhancement_ppm": float(peak_ppm),
            "method": method,
            "true_threshold_ppm": true_threshold,
            "predicted_pixels_local": int(np.sum(predicted & evaluation_mask)),
            **binary_detection_metrics(true_mask, predicted, evaluation_mask),
        })

    for method, score in score_maps.items():
        continuous_metric_rows.append({
            "peak_enhancement_ppm": float(peak_ppm),
            "score": method,
            **continuous_score_metrics(true_mask, score, evaluation_mask),
        })

    annulus_mask = evaluation_mask & (~true_mask)
    for band_name, score in {"1.6 µm": z16, "2.3 µm": z23}.items():
        localization_error, detected_peak = peak_localization_error(
            score,
            injection_center_yx,
            evaluation_mask,
        )
        band_summary_rows.append({
            "peak_enhancement_ppm": float(peak_ppm),
            "band": band_name,
            "median_z_inside_true_plume": float(np.nanmedian(score[true_mask])),
            "max_z_inside_true_plume": float(np.nanmax(score[true_mask])),
            "median_z_local_background": float(np.nanmedian(score[annulus_mask])),
            "contrast_median_z": float(
                np.nanmedian(score[true_mask]) - np.nanmedian(score[annulus_mask])
            ),
            "peak_localization_error_pixels": localization_error,
            "detected_peak_row": detected_peak[0] if detected_peak else np.nan,
            "detected_peak_col": detected_peak[1] if detected_peak else np.nan,
        })

    true_values_16 = z16[true_mask]
    true_values_23 = z23[true_mask]
    finite_pair = np.isfinite(true_values_16) & np.isfinite(true_values_23)
    correlation = (
        float(np.corrcoef(true_values_16[finite_pair], true_values_23[finite_pair])[0, 1])
        if finite_pair.sum() >= 3 else np.nan
    )
    run_summary_rows.append({
        "peak_enhancement_ppm": float(peak_ppm),
        "true_pixels": int(true_mask.sum()),
        "evaluation_pixels": int(evaluation_mask.sum()),
        "new_candidate_16_pixels": int(new16.sum()),
        "new_candidate_23_pixels": int(new23.sum()),
        "new_dual_nearby_pixels": int(new_nearby.sum()),
        "fraction_of_new_23_supported_by_16": (
            float(np.sum(new_nearby) / np.sum(new23)) if np.sum(new23) else np.nan
        ),
        "z16_z23_correlation_in_true_plume": correlation,
        "background_pixels_16": int(result_16["background_mask"].sum()),
        "background_pixels_23": int(result_23["background_mask"].sum()),
    })

    if np.isclose(peak_ppm, PRIMARY_PEAK_PPM):
        primary_result = {
            "peak_ppm": float(peak_ppm),
            "enhancement_map": enhancement_map,
            "cube_injected": cube_injected,
            "ratio_cube": ratio_cube,
            "true_mask": true_mask,
            "evaluation_mask": evaluation_mask,
            "alpha16": alpha16,
            "alpha23": alpha23,
            "z16": z16,
            "z23": z23,
            "delta_z16": z16 - baseline_z_16,
            "delta_z23": z23 - baseline_z_23,
            "new16": new16,
            "new23": new23,
            "new_exact": new_exact,
            "new_nearby": new_nearby,
            "local_max_new_z16": local_max_new_z16,
            "result_16": result_16,
            "result_23": result_23,
        }


discrete_metrics_df = pd.DataFrame(discrete_metric_rows)
continuous_metrics_df = pd.DataFrame(continuous_metric_rows)
band_summary_df = pd.DataFrame(band_summary_rows)
run_summary_df = pd.DataFrame(run_summary_rows)

if primary_result is None:
    raise RuntimeError("PRIMARY_PEAK_PPMに対応する評価結果がありません。")

display(discrete_metrics_df)
display(continuous_metrics_df)
display(band_summary_df)
display(run_summary_df)
